## STFPM 비교 2차 (v3) — 버그 수정 + 우리 전처리 + L4 제약 해제

v3에서 바뀐 점: 학습 진행 막대는 fold 폴더의 `train_log.txt`로 보내고, 화면에는 결과만 출력한다. 출력이 너무 길면 탭이 끊길 수 있어서다. ②'는 이미 끝났다. 5번 셀을 실행하면 끝난 fold를 건너뛰고 바로 요약을 보여 준다. 중간에 끊긴 실험도 셀을 다시 실행하면 이어서 돈다.

1차(v1)에서 ② fold 0의 학습 폴더에 ①의 사진이 섞이는 버그가 있었다. 실험마다 작업 폴더를 따로 쓰도록 고쳤다. 전부 시간 블록 5-fold로 돌린다.

| 실험 | ②에서 바꾼 것 | 알 수 있는 것 |
| --- | --- | --- |
| ②' STFPM(팀원 A 설정) | 버그 수정만 | 깨끗한 기준 결과 |
| ④ STFPM + 우리 전처리 | 입력: 고정 크롭 4,000장 → 이동·마스킹 크롭 3,425장(자동 선별) | 판정 가능한 사진만 넣으면 STFPM도 나아지는지 |
| ⑤ STFPM + L4 설정 | 제약 해제: resnet18→34, 배치 16→32, epoch 15→50 | 제약을 푸는 효과 |

**준비:** Drive `산공종설/`에 아래 세 파일이 있어야 한다.
- `underfill_originals_4000.zip`: 원본, 1차 때 올린 것
- `keep3425_masked_500.zip`: 마스킹 크롭, 처음 지도학습 때 올린 것
- `stfpm_compare_bundle_v3.zip`: **새로 올릴 것**. 팀원 A 코드(수정 없음), 분할표, 고친 실행 스크립트

런타임은 **L4**. 5~7번은 합쳐서 2~3시간 걸릴 수 있다(추정).

## 1. GPU 확인

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "GPU 런타임이 아님: 런타임 > 런타임 유형 변경 > L4"

## 2. Drive 연결
`force_remount=True`는 웹에서 방금 올린 파일도 보이도록 Drive를 다시 연결한다.

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
import os
DRIVE_DIR = "/content/drive/MyDrive/산공종설"
RESULTS_DIR = f"{DRIVE_DIR}/results"
for f in ("underfill_originals_4000.zip", "keep3425_masked_500.zip", "stfpm_compare_bundle_v3.zip"):
    print("OK  " if os.path.exists(f"{DRIVE_DIR}/{f}") else "없음", f)

## 3. 복사 → 압축 해제
- `/content/orig`: 원본 4,000장. ②'와 ⑤에서 쓴다.
- `/content/data`: 마스킹 크롭 3,425장과 그 분할표(`folds.csv`). ④에서 쓴다. 처음 지도학습 때와 같은 사진, 같은 분할이다.
- `/content/bundle`: 팀원 A 코드와 실행 스크립트

마지막 두 줄은 각각 4000, 3425가 나와야 한다.

In [ ]:
!rm -rf /content/bundle
!cp "{DRIVE_DIR}/underfill_originals_4000.zip" "{DRIVE_DIR}/keep3425_masked_500.zip" "{DRIVE_DIR}/stfpm_compare_bundle_v3.zip" /content/
!unzip -q -o /content/underfill_originals_4000.zip -d /content/orig
!unzip -q -o /content/keep3425_masked_500.zip -d /content/data
!unzip -q -o /content/stfpm_compare_bundle_v3.zip -d /content/bundle
!grep -c "relaxed for L4" /content/bundle/stfpm_timeblock.py
!find /content/orig -name "*.jpg" | wc -l
!find /content/data/size_500/masked_crops -name "*.jpg" | wc -l

## 4. anomalib 설치
"세션 다시 시작" 안내가 뜨면 다시 시작한다. 그다음 **2~3번 셀을 다시 실행**한다. 변수는 사라지지만 파일과 패키지는 남는다.

In [ ]:
!pip install -q "anomalib==2.6.2" timm
import importlib.metadata as md, torch
print({p: md.version(p) for p in ("anomalib", "torch", "lightning")}, "| cuda", torch.cuda.is_available())

## 5. ②' STFPM 팀원 A 설정, 시간 블록 (버그 수정 재실행)
1차 ②와 설정이 같다. 결과는 새 폴더(`..._v2`)에 저장해서 1차 결과와 섞이지 않게 한다.

In [ ]:
!python /content/bundle/stfpm_timeblock.py --project /content/bundle/project_STFPM --orig-root /content/orig --split-csv /content/bundle/folds_all4000.csv --out "{RESULTS_DIR}/stfpm_resnet18_timeblock_all4000_v2"

## 6. ④ STFPM + 우리 전처리 (마스킹 크롭 3,425장)
처음 지도학습 결과(정확도 0.976, AUROC 0.99)와 **같은 사진, 같은 분할**이다. 방식만 STFPM이다.
- `--path-column masked_crop --orig-root /content/data`: 원본 대신 이미 잘라 둔 마스킹 크롭을 쓴다.
- `--no-roi-crop`: 이미 잘라 둔 사진이라 팀원 A 코드의 고정 위치 자르기를 끈다. 크기 조정(512)은 팀원 A 설정 그대로다.

②'보다 좋아지면, 베드·모서리 프레임이 STFPM의 발목을 잡았다는 가설이 맞는 거다.

In [ ]:
!python /content/bundle/stfpm_timeblock.py --project /content/bundle/project_STFPM --orig-root /content/data --path-column masked_crop --no-roi-crop --split-csv /content/data/folds.csv --out "{RESULTS_DIR}/stfpm_resnet18_keep3425_masked"

## 7. ⑤ STFPM + L4 설정 (제약 해제)
②'와 사진·분할이 같고, 팀원 A `train.py`에 박혀 있던 GTX 1660용 제한만 푼다.
- `--relax`: 팀원 A 코드를 **복사한 사본**에서 네 줄만 바꾼다(백본 제한, 배치 16 상한 두 줄, epoch 20 상한). 원본 코드는 그대로다.
- `--backbone resnet34 --batch-size 32 --max-epochs 50`: L4에서 가능한 설정 묶음이다.

메모리가 부족하다는 에러(`CUDA out of memory`)가 나면 `--batch-size 16`으로 줄인다. 이미 끝난 fold는 건너뛴다.

In [ ]:
!python /content/bundle/stfpm_timeblock.py --project /content/bundle/project_STFPM --orig-root /content/orig --split-csv /content/bundle/folds_all4000.csv --relax --backbone resnet34 --batch-size 32 --max-epochs 50 --out "{RESULTS_DIR}/stfpm_resnet34_b32_e50_timeblock_all4000"

## 8. 비교표
- STFPM의 `auroc`는 fold별 AUROC의 평균이다. `accuracy` 등은 팀원 A 방식 기준값(정상 학습 점수 상위 1%)으로 판정한 결과다.
- 지도학습은 확률 0.5를 기준으로 판정한다.
- 1차 ①·③ 결과도 같이 보여 준다.

이 셀 출력을 그대로 Claude에게 붙여 주면 된다.

In [ ]:
import json, pandas as pd
rows, folds = [], {}
def stfpm(name, sub):
    s = json.load(open(f"{RESULTS_DIR}/{sub}/summary.json"))
    p = s["pooled"]
    rows.append({"run": name, "accuracy": p["accuracy"], "underfill_recall": p["underfill_recall"],
                 "normal_recall": p["normal_recall"], "auroc": s["fold_mean_auroc"], "n_test": p["n"]})
    folds[name] = {k: round(v["auroc"], 3) for k, v in s["per_fold"].items()}
def supervised(name, sub):
    s = pd.read_csv(f"{RESULTS_DIR}/{sub}/summary.csv", encoding="utf-8-sig")
    for _, r in s.iterrows():
        rows.append({"run": f"{name} ({r['mode']})", "accuracy": r["pooled_mean_accuracy"],
                     "underfill_recall": r["pooled_mean_underfill_recall"], "normal_recall": r["pooled_mean_normal_recall"],
                     "auroc": r["pooled_mean_auroc"], "n_test": None})
for name, sub, fn in (("① STFPM 팀원 A 분할(1차)", "stfpm_resnet18_stride5", stfpm),
                      ("②' STFPM 시간블록", "stfpm_resnet18_timeblock_all4000_v2", stfpm),
                      ("④ STFPM 마스킹·선별", "stfpm_resnet18_keep3425_masked", stfpm),
                      ("⑤ STFPM L4설정", "stfpm_resnet34_b32_e50_timeblock_all4000", stfpm),
                      ("③ 지도 고정크롭(1차)", "resnet18_fixedcrop512_all4000_fast_seeds0", supervised),
                      ("기존 지도 마스킹·선별", "resnet18_keep3425_masked_500_fast_seeds0", supervised)):
    try:
        fn(name, sub)
    except FileNotFoundError:
        print("아직 없음:", name)
print(pd.DataFrame(rows).round(3).to_string(index=False))
print("\nSTFPM fold별 AUROC:")
for k, v in folds.items():
    print(" ", k, v)

## 9. 런타임 반납
자리를 비울 거면 5~8번과 함께 이 셀도 실행 대기시켜 둔다.

In [ ]:
from google.colab import drive, runtime
drive.flush_and_unmount()
runtime.unassign()